# Transformation Walkthrough
Step-by-step transformation logic for each data source.

Follow the journey from raw extracted data through each transformation layer:
schema standardisation, CRM cleaning, unified ads, web channel grouping,
and email engagement normalisation.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.3f}'.format)

from datetime import datetime
from config import BASE_DIR

# Extract a small slice of raw data for demonstration
from src.extractors import CRMExtractor, MetaAdsExtractor, GoogleAdsExtractor, GA4Extractor, EmailPlatformExtractor

END   = datetime(2025, 3, 1)
START = datetime(2025, 1, 1)

crm_ext   = CRMExtractor({"accounts_count": 100, "contacts_count": 300,
                           "opportunities_count": 60, "activities_count": 400})
meta_ext  = MetaAdsExtractor({"campaigns": 4})
gads_ext  = GoogleAdsExtractor({"campaigns": 3})
ga4_ext   = GA4Extractor({"sessions_count": 1000, "events_count": 4000})
email_ext = EmailPlatformExtractor({"campaigns": 6, "sends_count": 1000})

crm_raw   = crm_ext.extract(START, END)
meta_raw  = meta_ext.extract(START, END)
gads_raw  = gads_ext.extract(START, END)
ga4_raw   = ga4_ext.extract(START, END)
email_raw = email_ext.extract(START, END)

print("Raw data ready for transformation demonstration.")


In [ ]:
from src.transformers import SchemaStandardizer

standardizer = SchemaStandardizer()

# Isolate CRM contacts for demonstration
contacts_raw = crm_ext.contacts_df.copy()
print("=== Before SchemaStandardizer ===")
print("Columns:", contacts_raw.columns.tolist())
print()
display(contacts_raw.head(3))

contacts_std = standardizer.transform(contacts_raw, source_type="crm")
print()
print("=== After SchemaStandardizer ===")
print("Columns:", contacts_std.columns.tolist())
print()
display(contacts_std.head(3))

# Highlight column name changes
orig_cols = set(contacts_raw.columns)
new_cols  = set(contacts_std.columns)
added     = new_cols - orig_cols
removed   = orig_cols - new_cols
print(f"\nColumns added   : {sorted(added)}")
print(f"Columns removed : {sorted(removed)}")
print(f"Metadata: {standardizer.metadata}")


In [ ]:
from src.transformers import CRMTransformer

crm_transformer = CRMTransformer()

print("=== CRM Contacts – Before ===")
print("lead_status value counts:")
display(contacts_std['lead_status'].value_counts().rename("count").to_frame())

contacts_t = crm_transformer.transform(contacts_std)

print()
print("=== CRM Contacts – After CRMTransformer ===")
print("lead_status value counts (normalised):")
display(contacts_t['lead_status'].value_counts().rename("count").to_frame())

print()
print("New derived columns:")
new_cols = [c for c in contacts_t.columns if c not in contacts_std.columns]
print(new_cols)

print()
print("Sample of derived fields (first 5 rows):")
derived_sample_cols = [c for c in ['lead_status', 'email_valid', 'account_age_days',
                                    'days_since_last_activity', 'lead_age_days'] if c in contacts_t.columns]
display(contacts_t[derived_sample_cols].head())
print(f"\nTransformer metadata: {crm_transformer.metadata}")


In [ ]:
from src.transformers import AdsTransformer

ads_transformer = AdsTransformer()

print("=== Unified Ads Transformation ===")
print(f"Meta raw rows  : {len(meta_raw):,}")
print(f"Google raw rows: {len(gads_raw):,}")

unified_ads = ads_transformer.transform(meta_data=meta_raw, google_data=gads_raw)

print(f"\nUnified rows   : {len(unified_ads):,}")
print(f"Columns: {unified_ads.columns.tolist()}")
print()
display(unified_ads.head(4))

# Platform breakdown
platform_summary = (
    unified_ads.groupby('platform')
    .agg(rows=('spend', 'count'),
         total_spend=('spend', 'sum'),
         total_clicks=('clicks', 'sum'),
         total_conversions=('conversions', 'sum'))
    .round(2)
)
print("\nPlatform summary:")
display(platform_summary)


In [ ]:
# ── Channel grouping logic ─────────────────────────────────────────────────
from config import CHANNEL_GROUPING

print("=== Channel Grouping Rules ===")
for channel, rules in CHANNEL_GROUPING.items():
    print(f"  {channel:20s}: sources={rules.get('sources', [])}, mediums={rules.get('mediums', [])}")

print()
print("=== Unified Ads: Channel Distribution ===")
channel_dist = unified_ads['channel'].value_counts().rename("row_count")
display(channel_dist.to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
channel_dist.plot(kind='barh', ax=ax, color=['#1877F2', '#4285F4', '#34A853', '#FBBC05'])
ax.set_title("Unified Ads: Row Count by Channel")
ax.set_xlabel("Rows")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
from src.transformers import WebTransformer

web_transformer = WebTransformer()

# GA4 extractor stores sessions separately
ga4_sessions = ga4_raw.copy() if 'record_type' not in ga4_raw.columns                else ga4_raw[ga4_raw['record_type'] == 'session'].copy()

print("=== Web Session Transformation ===")
print(f"Input rows: {len(ga4_sessions):,}")

web_t = web_transformer.transform(ga4_sessions)

print(f"Output rows: {len(web_t):,}")
print()

# Engagement score demonstration
if 'engagement_score' in web_t.columns:
    print("Engagement score statistics:")
    display(web_t['engagement_score'].describe().rename("engagement_score"))

    print()
    print("Engagement score = (pages_per_session × 10 + session_duration_s / 10) / 2")
    sample_cols = [c for c in ['source', 'medium', 'channel_group',
                                'pages_per_session', 'session_duration_seconds',
                                'engagement_score', 'is_bot'] if c in web_t.columns]
    display(web_t[sample_cols].head(8))

print(f"\nBot sessions flagged: {web_t['is_bot'].sum():,}" if 'is_bot' in web_t.columns else "")
print(f"Transformer metadata: {web_transformer.metadata}")


In [ ]:
# ── Transformation metadata – rows in vs rows out ─────────────────────────
from src.transformers import EmailTransformer

email_transformer = EmailTransformer()
email_t = email_transformer.transform(email_raw)

summary_rows = [
    {"Stage": "CRM contacts → SchemaStandardizer",
     "Rows In": len(contacts_raw), "Rows Out": len(contacts_std),
     "Delta": len(contacts_std) - len(contacts_raw)},
    {"Stage": "CRM contacts → CRMTransformer",
     "Rows In": len(contacts_std), "Rows Out": len(contacts_t),
     "Delta": len(contacts_t) - len(contacts_std)},
    {"Stage": "Meta + Google → AdsTransformer",
     "Rows In": len(meta_raw) + len(gads_raw), "Rows Out": len(unified_ads),
     "Delta": len(unified_ads) - len(meta_raw) - len(gads_raw)},
    {"Stage": "GA4 sessions → WebTransformer",
     "Rows In": len(ga4_sessions), "Rows Out": len(web_t),
     "Delta": len(web_t) - len(ga4_sessions)},
    {"Stage": "Email → EmailTransformer",
     "Rows In": len(email_raw), "Rows Out": len(email_t),
     "Delta": len(email_t) - len(email_raw)},
]

summary_df = pd.DataFrame(summary_rows)
print("=== Transformation Metadata (rows in vs rows out) ===")
display(summary_df.set_index("Stage"))
